In [ ]:
# Setup
%load_ext autoreload
%autoreload 2

import sys
import os

# Add parent directory to path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
project_root = os.path.dirname(notebook_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt

# Import NEW unified API
from src.dataset import PINNDataset
from src.train_loop import train_pinn
from src.visualization import plot_results, plot_losses

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("✓ Modules loaded successfully")

In [ ]:
# Choose your dataset:
# Option 1: Calhoun CCZO (6 depths, no WTD)
config_file = '../configs/baseline.yaml'

# Option 2: US-Uaf (5 depths, with WTD)
# config_file = '../configs/us_uaf_2019.yaml'

print("=" * 70)
print("LOADING DATASET FROM YAML")
print("=" * 70)
print(f"Config: {config_file}\n")

# 🚀 ONE LINE to load everything!
dataset = PINNDataset(config_file, verbose=True)

In [ ]:
print("\n" + "=" * 70)
print("TRAINING PINN MODEL")
print("=" * 70)

# 🚀 ONE LINE to train!
model, losses, comps, sample_losses, sample_comps, sample_epochs = train_pinn(
    dataset,
    device=device
)

print("\n" + "=" * 70)
print("TRAINING COMPLETE!")
print("=" * 70)
print(f"Final loss: {losses[-1]:.3e}")

In [ ]:
print("\n" + "=" * 70)
print("VISUALIZING RESULTS")
print("=" * 70)

# 🎨 ONE LINE to plot everything!
# Automatically:
# - Uses all available observation depths
# - Adds WTD if available
# - Formats labels correctly
plot_results(model, dataset, device=device)
plt.show()

print("\n✓ Comprehensive results plotted")

In [ ]:
# 📊 Plot training losses
plot_losses(losses, comps, sample_losses, sample_comps, sample_epochs)
plt.show()

print("\n✓ Training losses plotted")

In [ ]:
print("Dataset Information:")
print(f"  Config: {dataset.config_path}")
print(f"  Data source: {dataset.config.data_path}")
print(f"  Date range: {dataset.config.start_date} to {dataset.config.end_date}")
print(f"\n  Observation depths: {dataset.obs_depths}")
print(f"  Total time points: {len(dataset.obs_times)}")
print(f"  Duration: {dataset.obs_times[-1]/86400:.1f} days")
print(f"\n  Boundary condition:")
print(f"    Type: {dataset.bc_type}")
print(f"    Points: {len(dataset.bc_times)}")
print(f"    Range: {dataset.bc_values.min():.4f} - {dataset.bc_values.max():.4f} m³/m³")
print(f"\n  Initial condition:")
print(f"    Type: {dataset.ic_type}")
print(f"    Profile points: {len(dataset.ic_profile['depths'])}")
print(f"\n  Water table depth: {'Available' if dataset.has_wtd() else 'Not available'}")
print(f"\n  Training:")
print(f"    Epochs: {dataset.n_epochs}")
print(f"    Learning rate: {dataset.learning_rate}")
print(f"    Device: {dataset.device}")